# Calculate daily China AQI and pollutant classifications

This notebook loads hourly readings from the `all_regions` table in the MySQL `air_quality` database. It calculates one daily AQI and classification per station and calendar day, then creates reproducible stratified training and test sets. MySQL is required to load the source table; database writes for the split tables remain optional.


## China AQI basis and six-level classification

This notebook uses China’s AQI breakpoints, not WHO guideline values. The Ministry response linked below explains the HJ 633-2012 rationale; the current technical regulation is HJ 633-2026.

The output has one row per station and calendar day. PM2.5, PM10, SO2, NO2, and CO use 24-hour daily means, requiring at least 20 valid hourly readings for each pollutant. O3 uses the day’s maximum trailing 8-hour average; each window requires 6 valid readings and a day requires at least 14 valid windows ending from 08:00 through 23:00. When the daily maximum 8-hour O3 concentration exceeds 800 µg/m³, the daily maximum 1-hour O3 concentration is used for its IAQI. CO source values are converted from µg/m³ to mg/m³.

Each pollutant IAQI is linearly interpolated from its concentration breakpoints. The overall daily AQI is the maximum available pollutant IAQI.

| Category code | AQI range | China category |
| ---: | ---: | --- |
| 1 | 0–50 | Excellent |
| 2 | 51–100 | Good |
| 3 | 101–150 | Lightly polluted |
| 4 | 151–200 | Moderately polluted |
| 5 | 201–300 | Heavily polluted |
| 6 | 301–500 | Severely polluted |

| AQI breakpoint | PM2.5 | PM10 | SO2 (24h) | NO2 (24h) | CO (24h) | O3 (8h) |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 50 | 30 | 50 | 50 | 40 | 2 | 100 |
| 100 | 60 | 120 | 150 | 80 | 4 | 160 |
| 150 | 115 | 250 | 475 | 180 | 14 | 215 |
| 200 | 150 | 350 | 800 | 280 | 24 | 265 |
| 300 | 250 | 420 | 1600 | 565 | 36 | 800 |
| 400 | 350 | 500 | 2100 | 750 | 48 | — |
| 500 | 500 | 600 | 2620 | 940 | 60 | — |

Concentration units are µg/m³ except CO, which is mg/m³. The input must contain no duplicate station-hour records. Sources: [Ministry of Ecology and Environment, linked response](https://www.mee.gov.cn/xxgk2018/xxgk/xxgk13/201810/t20181015_662213_wh.html), [HJ 633-2026 standard page](https://www.mee.gov.cn/ywgz/fgbz/bz/bzwb/jcffbz/202602/t20260225_1144441.shtml).

## Create reproducible stratified training and test sets

An 80/20 split is made separately for each station in the complete cleaned source table. This preserves the station proportions and keeps all source columns and rows; it does not use the daily AQI table or require AQI labels. The sets remain in memory as `train_set` and `test_set`.


In [1]:
import os
import re
import pandas as pd
from sqlalchemy import create_engine, text

# Uses this local default if AIR_QUALITY_DB_URL is not set.
# Set AIR_QUALITY_DB_URL to override it when your MySQL credentials differ.
DB_URL = os.environ.get(
    "AIR_QUALITY_DB_URL",
    "mysql+pymysql://root:@localhost/air_quality",
)
SOURCE_TABLE = "all_regions"
WRITE_SPLITS_TO_DB = False

if not re.fullmatch(r"[A-Za-z0-9_]+", SOURCE_TABLE):
    raise ValueError(f"Invalid MySQL table name: {SOURCE_TABLE!r}")

engine = create_engine(DB_URL, pool_pre_ping=True)
query = text(f"SELECT * FROM {SOURCE_TABLE}")

try:
    with engine.connect() as connection:
        source_df = pd.read_sql_query(query, connection)
except Exception as error:
    raise RuntimeError(
        f"Could not load table {SOURCE_TABLE!r} from the air_quality database. "
        "Check that MySQL is running, the database and table exist, "
        "AIR_QUALITY_DB_URL has the correct credentials, and PyMySQL is installed. "
        f"Original database error: {error}"
    ) from error

print(f"Loaded {len(source_df):,} rows from air_quality.{SOURCE_TABLE}")
display(source_df.head())


Loaded 418,753 cleaned rows from c:\Users\PLP\Downloads\Data_Science\Data_Science\Notebooks\cleaned_dataset_validated.csv


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,interpolated_columns
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin,NaN
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin,NaN
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin,NaN
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin,NaN
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin,NaN


## Calculate daily averages and China AQI values

Generate one result per station and calendar day. Daily means for PM2.5, PM10, SO2, NO2, and CO require at least 20 valid hourly readings. For O3, use the day’s maximum valid trailing 8-hour average (at least 6 readings per window and at least 14 valid windows ending from 08:00 through 23:00). Convert daily concentrations to pollutant IAQIs, take the maximum available IAQI as the daily AQI, and assign its China category.

In [2]:
# HJ 633-2026 daily concentration breakpoints; concentrations are ug/m3,
# except CO in mg/m3. AQI is calculated once per station and calendar day.
AQI_BREAKPOINTS = [0, 50, 100, 150, 200, 300, 400, 500]
POLLUTANT_BREAKPOINTS = {
    "PM2.5": [0, 30, 60, 115, 150, 250, 350, 500],
    "PM10": [0, 50, 120, 250, 350, 420, 500, 600],
    "SO2": [0, 50, 150, 475, 800, 1600, 2100, 2620],
    "NO2": [0, 40, 80, 180, 280, 565, 750, 940],
    "CO": [0, 2, 4, 14, 24, 36, 48, 60],
    "O3": [0, 100, 160, 215, 265, 800],
}
O3_1H_BREAKPOINTS = [0, 160, 200, 300, 400, 800, 1000, 1200]
AQI_CATEGORIES = {
    1: "Excellent", 2: "Good", 3: "Lightly polluted",
    4: "Moderately polluted", 5: "Heavily polluted", 6: "Severely polluted",
}
DAILY_POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO"]

required_columns = [*POLLUTANT_BREAKPOINTS, "station", "year", "month", "day", "hour"]
missing_columns = [column for column in required_columns if column not in source_df.columns]
if missing_columns:
    raise KeyError(f"Required pollutant columns are missing from {SOURCE_TABLE!r}: {missing_columns}")

hourly_df = source_df.copy()
time_parts = hourly_df[["year", "month", "day", "hour"]].apply(pd.to_numeric, errors="coerce")
timestamps = pd.to_datetime(
    {"year": time_parts["year"], "month": time_parts["month"], "day": time_parts["day"], "hour": time_parts["hour"]},
    errors="coerce",
)
hourly_df["_timestamp"] = timestamps
hourly_df["_date"] = timestamps.dt.normalize()
hourly_df["_station_key"] = hourly_df["station"].astype("string").str.strip().str.casefold()
hourly_df = hourly_df.loc[
    hourly_df["_timestamp"].notna() & hourly_df["_station_key"].notna()
].copy()
if hourly_df.duplicated(["_station_key", "_timestamp"]).any():
    raise ValueError("Input contains duplicate station-hour rows; resolve duplicates before daily AQI calculation.")
for pollutant in POLLUTANT_BREAKPOINTS:
    readings = pd.to_numeric(hourly_df[pollutant], errors="coerce")
    hourly_df[pollutant] = readings.mask(readings.lt(0))

group_keys = ["_station_key", "_date"]
daily_group = hourly_df.groupby(group_keys, sort=False)
labeled_df = daily_group[["station"]].first()
labeled_df.index.names = group_keys
for pollutant in DAILY_POLLUTANTS:
    valid_hour_count = daily_group[pollutant].count()
    daily_mean = daily_group[pollutant].mean()
    labeled_df[f"{pollutant}_24h_avg"] = daily_mean.where(valid_hour_count.ge(20))

# Daily ozone is the maximum valid trailing 8-hour average for each date.
o3_rolling_parts = []
for _, station_rows in hourly_df.groupby("_station_key", sort=False):
    ordered = station_rows.sort_values("_timestamp")
    o3_series = pd.Series(
        ordered["O3"].to_numpy(), index=pd.DatetimeIndex(ordered["_timestamp"])
    )
    o3_rolling = o3_series.rolling("8h", closed="right", min_periods=6).mean()
    o3_rolling_parts.append(pd.DataFrame({
        "_station_key": ordered["_station_key"].to_numpy(),
        "_date": ordered["_date"].to_numpy(),
        "_hour": ordered["_timestamp"].dt.hour.to_numpy(),
        "O3_8h_avg": o3_rolling.to_numpy(),
    }))
o3_windows = pd.concat(o3_rolling_parts, ignore_index=True)
o3_windows = o3_windows.loc[o3_windows["_hour"].between(8, 23)]
o3_daily = o3_windows.groupby(group_keys, sort=False)["O3_8h_avg"]
o3_daily_max = o3_daily.max()
o3_valid_count = o3_daily.count()
labeled_df["O3_8h_avg"] = o3_daily_max.where(o3_valid_count.ge(14))
labeled_df["O3_1h_max"] = daily_group["O3"].max()
labeled_df["date"] = labeled_df.index.get_level_values("_date")
labeled_df = labeled_df.reset_index(drop=True)
labeled_df["CO_mg_m3"] = labeled_df["CO_24h_avg"] * 0.001

def interpolate_iaqi(concentration, concentration_breakpoints):
    """Interpolate concentrations between the supplied China AQI breakpoints."""
    result = pd.Series(float("nan"), index=concentration.index, dtype="float64")
    for index in range(len(concentration_breakpoints) - 1):
        bp_lo, bp_hi = concentration_breakpoints[index:index + 2]
        iaqi_lo, iaqi_hi = AQI_BREAKPOINTS[index:index + 2]
        in_range = concentration.gt(bp_lo) & concentration.le(bp_hi)
        result.loc[in_range] = (
            (iaqi_hi - iaqi_lo) / (bp_hi - bp_lo)
            * (concentration.loc[in_range] - bp_lo) + iaqi_lo
        )
    result.loc[concentration.eq(0)] = 0
    return result

for pollutant in DAILY_POLLUTANTS:
    concentration = labeled_df["CO_mg_m3"] if pollutant == "CO" else labeled_df[f"{pollutant}_24h_avg"]
    pollutant_aqi = interpolate_iaqi(concentration, POLLUTANT_BREAKPOINTS[pollutant])
    labeled_df[f"{pollutant}_iaqi"] = pollutant_aqi.round().astype("Int64")

o3_concentration = labeled_df["O3_8h_avg"]
o3_aqi = interpolate_iaqi(o3_concentration, POLLUTANT_BREAKPOINTS["O3"])
# Above 800 ug/m3, use the daily maximum 1-hour O3 concentration.
o3_1h_aqi = interpolate_iaqi(labeled_df["O3_1h_max"], O3_1H_BREAKPOINTS)
use_o3_1h = o3_concentration.gt(800)
o3_aqi.loc[use_o3_1h] = o3_1h_aqi.loc[use_o3_1h]
labeled_df["O3_iaqi"] = o3_aqi.round().astype("Int64")

iaqi_columns = [f"{pollutant}_iaqi" for pollutant in POLLUTANT_BREAKPOINTS]
labeled_df["aqi"] = labeled_df[iaqi_columns].max(axis=1, skipna=True).astype("Int64")
labeled_df.loc[labeled_df[iaqi_columns].isna().all(axis=1), "aqi"] = pd.NA
labeled_df["aqi_category_code"] = pd.Series(pd.NA, index=labeled_df.index, dtype="Int64")
for code, (lower, upper) in enumerate([(0, 50), (51, 100), (101, 150), (151, 200), (201, 300), (301, 500)], 1):
    labeled_df.loc[labeled_df["aqi"].between(lower, upper), "aqi_category_code"] = code
labeled_df["aqi_category"] = labeled_df["aqi_category_code"].map(AQI_CATEGORIES)

print("Daily pollutant IAQI counts:")
for pollutant in POLLUTANT_BREAKPOINTS:
    print(f"{pollutant}:")
    display(labeled_df[f"{pollutant}_iaqi"].value_counts(dropna=True).sort_index().rename("station-days"))
print("\nDaily China AQI category counts:")
display(labeled_df["aqi_category"].value_counts().reindex(AQI_CATEGORIES.values(), fill_value=0).rename("station-days"))
print(f"Daily station records: {len(labeled_df):,}")

# Attach each station-day's daily AQI values to every original hourly row
# from that station and date, preserving the original hourly row count.
daily_labeled_df = labeled_df.copy()
daily_labeled_df["_station_key"] = daily_labeled_df["station"].astype("string").str.strip().str.casefold()
daily_labeled_df["_date"] = pd.to_datetime(daily_labeled_df["date"])
daily_labeled_df = daily_labeled_df.rename(columns={
    **{f"{pollutant}_24h_avg": f"{pollutant}_daily_avg" for pollutant in DAILY_POLLUTANTS},
    "O3_8h_avg": "O3_daily_max_8h_avg",
    "O3_1h_max": "O3_daily_max_1h_avg",
    "CO_mg_m3": "CO_daily_avg_mg_m3",
})
daily_value_columns = [
    *[f"{pollutant}_daily_avg" for pollutant in DAILY_POLLUTANTS],
    "O3_daily_max_8h_avg", "O3_daily_max_1h_avg", "CO_daily_avg_mg_m3",
    *iaqi_columns, "aqi", "aqi_category_code", "aqi_category",
]
hourly_labeled_df = source_df.copy()
hourly_labeled_df["_row_order"] = range(len(hourly_labeled_df))
hourly_labeled_df["_station_key"] = hourly_labeled_df["station"].astype("string").str.strip().str.casefold()
hourly_time_parts = hourly_labeled_df[["year", "month", "day", "hour"]].apply(pd.to_numeric, errors="coerce")
hourly_timestamps = pd.to_datetime(
    {"year": hourly_time_parts["year"], "month": hourly_time_parts["month"],
     "day": hourly_time_parts["day"], "hour": hourly_time_parts["hour"]},
    errors="coerce",
)
hourly_labeled_df["_date"] = hourly_timestamps.dt.normalize()
hourly_labeled_df = hourly_labeled_df.merge(
    daily_labeled_df[["_station_key", "_date", *daily_value_columns]],
    on=["_station_key", "_date"],
    how="left",
    sort=False,
    validate="many_to_one",
)
hourly_labeled_df = hourly_labeled_df.sort_values("_row_order").drop(
    columns=["_row_order", "_station_key", "_date"]
).reset_index(drop=True)
labeled_df = hourly_labeled_df
print(f"Hourly rows with daily AQI labels: {labeled_df['aqi'].notna().sum():,} / {len(labeled_df):,}")

Daily pollutant IAQI counts:
PM2.5:


PM2.5_iaqi
5       3
6       6
7      11
8      27
9      36
       ..
491     2
496     1
497     1
499     1
500     1
Name: station-days, Length: 452, dtype: Int64

PM10:


PM10_iaqi
5       4
6       8
7      11
8      13
9      14
       ..
455     1
461     1
465     1
480     1
485     1
Name: station-days, Length: 327, dtype: Int64

SO2:


...

O3:


O3_iaqi
0        7
1      196
2      203
3      195
4      214
      ... 
212      2
215      2
253      1
255      1
436      1
Name: station-days, Length: 216, dtype: Int64


Daily China AQI category counts:


aqi_category
Excellent              2334
Good                   5042
Lightly polluted       5325
Moderately polluted    2388
Heavily polluted       1707
Severely polluted       592
Name: station-days, dtype: int64

Daily station records: 17,492
Hourly rows with daily AQI labels: 417,099 / 418,753


## Create reproducible stratified training and test sets

An 80/20 split is made separately for each station in the complete cleaned source table. This preserves the station proportions and keeps all source columns and rows; it does not use the daily AQI table or require AQI labels. The sets remain in memory as `train_set` and `test_set`.


In [ ]:
# Create reproducible 80/20 training and test sets from the complete cleaned source table.
# Stratify by station so each station is represented in both sets.

TEST_FRACTION = 0.20
RANDOM_STATE = 42

split_df = source_df.copy()
if "station" not in split_df.columns:
    raise KeyError("The source table must contain a 'station' column to stratify the split by station.")

train_parts = []
test_parts = []
for _, group in split_df.groupby("station", dropna=False, sort=False):
    if len(group) < 2:
        # A single-row station group cannot be present in both sets; retain it in training.
        train_parts.append(group)
        continue
    test_count = min(len(group) - 1, max(1, round(len(group) * TEST_FRACTION)))
    test_part = group.sample(n=test_count, random_state=RANDOM_STATE)
    train_parts.append(group.drop(index=test_part.index))
    test_parts.append(test_part)

train_set = (
    pd.concat(train_parts)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)
test_set = (
    pd.concat(test_parts)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
    if test_parts else split_df.iloc[0:0].copy()
)

print(f"Complete source rows: {len(split_df):,}")
print(f"Training set: {len(train_set):,} rows")
print(f"Test set: {len(test_set):,} rows")
print(f"Total after split: {len(train_set) + len(test_set):,} rows")
display(train_set["station"].value_counts(normalize=True, dropna=False).sort_index().rename("training_proportion"))
display(test_set["station"].value_counts(normalize=True, dropna=False).sort_index().rename("test_proportion"))

# Optional: save both sets to MySQL.
if WRITE_SPLITS_TO_DB:
    train_set.to_sql("training_set", engine, if_exists="replace", index=False)
    test_set.to_sql("test_set", engine, if_exists="replace", index=False)
    print("Saved training_set and test_set tables to air_quality")
else:
    print("Database writes disabled.")


## Optionally save the split tables in MySQL

The default is `False` to avoid changing the database unexpectedly. Set `WRITE_SPLITS_TO_DB = True` after reviewing the generated counts. The tables contain the original hourly pollutant values plus the matching station-day pollutant averages, IAQI values, daily AQI, and daily category. Use versioned table names instead if multiple experiments must be retained.


In [4]:
if WRITE_SPLITS_TO_DB:
    DB_URL = os.environ.get("AIR_QUALITY_DB_URL")
    if not DB_URL:
        raise RuntimeError("Set AIR_QUALITY_DB_URL before enabling database writes.")
    from sqlalchemy import create_engine
    engine = create_engine(DB_URL)
    train_set.to_sql("training_set", engine, if_exists="replace", index=False)
    test_set.to_sql("test_set", engine, if_exists="replace", index=False)
    print("Saved training_set and test_set tables to air_quality")
else:
    print("Database writes disabled. Set WRITE_SPLITS_TO_DB = True to save the split tables.")


Database writes disabled. Set WRITE_SPLITS_TO_DB = True to save the split tables.
